# Faruq-v3 — STB1 vs CMC0 object-level complementarity audit

Validation-only, post-training diagnostic. Tidak ada training dan test tidak diekstrak. Audit mencocokkan final detections ke GT secara class-agnostic pada IoU >= 0.50 lalu mengukur directional rescue, shared-error Jaccard, oracle headroom, per-class rescue, dan confusion-pair rescue untuk seeds 42/123/2026.

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import json, os, shutil, subprocess, sys, tarfile, time
from pathlib import Path

REPO = Path('/content/coffee-bean-detection')
BRANCH = 'agent/error-complementarity-audit-zzz'
if (REPO / '.git').is_dir():
    subprocess.run(['git', 'fetch', 'origin', BRANCH], cwd=REPO, check=True)
    subprocess.run(['git', 'checkout', BRANCH], cwd=REPO, check=True)
    subprocess.run(['git', 'reset', '--hard', f'origin/{BRANCH}'], cwd=REPO, check=True)
else:
    if REPO.exists():
        shutil.rmtree(REPO)
    clone = ['git', 'clone', '--depth', '1', '--branch', BRANCH, 'https://github.com/ediprin/coffee-bean-detection.git', str(REPO)]
    for attempt in range(1, 4):
        result = subprocess.run(clone)
        if result.returncode == 0:
            break
        if REPO.exists():
            shutil.rmtree(REPO)
        if attempt == 3:
            raise RuntimeError('Git clone gagal tiga kali.')
        time.sleep(2)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO)], check=True)
for key in list(sys.modules):
    if key == 'coffee_detector' or key.startswith('coffee_detector.'):
        sys.modules.pop(key, None)
sys.path.insert(0, str(REPO / 'src'))
os.chdir(REPO)
print('COMMIT:', subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO, text=True).strip())

In [ ]:
import torch
from coffee_detector.drive_project import require_project_artifact, resolve_drive_project_root

assert torch.cuda.is_available(), 'Aktifkan T4 GPU agar enam inference pass lebih cepat.'
REQUIRED = (
    'bundles/faruq-development-v3-grouped.tar',
    'experiments/faruq-v3-stb-paired-confirmation-v1/val_reports/stb_capacity_paired_confirmation.json',
    'experiments/faruq-v3-stb-paired-confirmation-v1/CMC0/CMC0_seed123/weights/best.pt',
    'experiments/faruq-v3-stb-paired-confirmation-v1/CMC0/CMC0_seed2026/weights/best.pt',
    'experiments/faruq-v3-stb-paired-confirmation-v1/STB1/STB1_seed123/weights/best.pt',
    'experiments/faruq-v3-stb-paired-confirmation-v1/STB1/STB1_seed2026/weights/best.pt',
)
PROJECT_ROOT = resolve_drive_project_root(required_relative_paths=REQUIRED)
ARCHIVE = require_project_artifact(PROJECT_ROOT, REQUIRED[0])
PAIRED_SUMMARY = require_project_artifact(PROJECT_ROOT, REQUIRED[1])
CMC0_123 = require_project_artifact(PROJECT_ROOT, REQUIRED[2])
CMC0_2026 = require_project_artifact(PROJECT_ROOT, REQUIRED[3])
STB_123 = require_project_artifact(PROJECT_ROOT, REQUIRED[4])
STB_2026 = require_project_artifact(PROJECT_ROOT, REQUIRED[5])

def unique_checkpoint(pattern):
    matches = sorted(PROJECT_ROOT.rglob(pattern))
    if len(matches) != 1:
        raise RuntimeError(f'Harus ada tepat satu {pattern}; ditemukan {len(matches)}: {matches[:10]}')
    return matches[0]

CMC0_42 = unique_checkpoint('CMC0_seed42/weights/best.pt')
STB_42 = unique_checkpoint('STB1_seed42/weights/best.pt')
DATA_ROOT = Path('/content/faruq-development-v3-grouped')
if not (DATA_ROOT / 'data.yaml').is_file():
    with tarfile.open(ARCHIVE, 'r') as archive:
        archive.extractall('/content', filter='data')
assert (DATA_ROOT / 'data.yaml').is_file()
assert not (DATA_ROOT / 'test').exists(), 'Test tidak boleh tersedia.'
OUTPUT_ROOT = PROJECT_ROOT / 'experiments/faruq-v3-stb-cmc0-complementarity-v1'
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print('GPU     :', torch.cuda.get_device_name(0))
print('PROJECT :', PROJECT_ROOT)
print('CMC0-42 :', CMC0_42)
print('STB1-42 :', STB_42)
print('OUTPUT  :', OUTPUT_ROOT)

## Jalankan audit

Audit menjalankan enam inference pass (CMC0/STB1 x 3 seed). Ini tidak mengubah checkpoint dan tidak melatih model.

In [ ]:
command = [
    sys.executable, '-u', '-m', 'coffee_detector.analysis.stb_cmc0_complementarity',
    '--data-root', str(DATA_ROOT),
    '--paired-summary', str(PAIRED_SUMMARY),
    '--output-root', str(OUTPUT_ROOT),
    '--cmc0-seed42', str(CMC0_42),
    '--cmc0-seed123', str(CMC0_123),
    '--cmc0-seed2026', str(CMC0_2026),
    '--stb-seed42', str(STB_42),
    '--stb-seed123', str(STB_123),
    '--stb-seed2026', str(STB_2026),
    '--device', '0',
]
print('MENJALANKAN:', ' '.join(command), flush=True)
subprocess.run(command, cwd=REPO, check=True)
print('AUDIT SELESAI.')

In [ ]:
import pandas as pd
from IPython.display import display

SUMMARY = OUTPUT_ROOT / 'stb_cmc0_complementarity.json'
assert SUMMARY.is_file(), SUMMARY
result = json.loads(SUMMARY.read_text(encoding='utf-8'))
assert result['evaluation_split'] == 'val'
assert result['test_images_accessed'] is False

agg_rows = []
for metric, value in result['aggregate'].items():
    if isinstance(value, dict) and {'mean', 'std'} <= set(value):
        agg_rows.append({'metric': metric, 'mean': value['mean'], 'std': value['std'], 'per_seed': value.get('values')})
display(pd.DataFrame(agg_rows).style.format({'mean': '{:.2%}', 'std': '{:.2%}'}))

seed_rows = []
for seed, row in result['per_seed'].items():
    seed_rows.append({
        'seed': seed,
        'CMC0 acc@IoU50': row['cmc0']['accuracy_iou50'],
        'STB acc@IoU50': row['stb1']['accuracy_iou50'],
        'STB-CMC0': row['stb_minus_cmc0_accuracy'],
        'CMC0->STB rescue': row['rescue']['cmc0_to_stb_rate_given_cmc0_error'],
        'STB->CMC0 rescue': row['rescue']['stb_to_cmc0_rate_given_stb_error'],
        'error Jaccard': row['error_overlap']['jaccard'],
        'oracle gain': row['oracle']['gain_over_best_model'],
    })
display(pd.DataFrame(seed_rows).style.format({c: '{:.2%}' for c in seed_rows[0] if c != 'seed'}))

print('TOP RESCUES PER SEED:')
for seed, row in result['per_seed'].items():
    print('\nSEED', seed)
    print('CMC0 wrong -> STB correct:', row['top_confusion_pair_rescues']['cmc0_wrong_stb_correct'][:10])
    print('STB wrong -> CMC0 correct:', row['top_confusion_pair_rescues']['stb_wrong_cmc0_correct'][:10])
print('\nSUMMARY:', SUMMARY)
print('Kirim tabel ini untuk interpretasi. Jangan membuka test.')